In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import joblib


# ============================================================
# Settings
# ============================================================
input_path = "results/summary/exid_metrics_summary_all_recordings_cleaned_selected_features_kmeans_clustered.csv"

output_dir = "results/logistic_regression"
os.makedirs(output_dir, exist_ok=True)

random_state = 42

target_col = "cluster"

numeric_features = [
    "max_pcad",
    "mean_pcad",
    "average_speed",
    "max_speed",
    "max_longitudinal_jerk",
    "max_lateral_jerk",
    "rms_total_jerk"
]

categorical_features = [
    "dominant_critical_position"
]


# ============================================================
# Load data
# ============================================================
df = pd.read_csv(input_path)

print("Loaded shape:", df.shape)
display(df.head())


# ============================================================
# Keep only successful rows if status column exists
# ============================================================
if "status" in df.columns:
    df = df[df["status"] == "ok"].copy()

print("Rows after status filtering:", len(df))


# ============================================================
# Check required columns
# ============================================================
required_cols = numeric_features + categorical_features + [target_col]

missing_cols = [col for col in required_cols if col not in df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.dropna(subset=[target_col]).copy()

# Make cluster target integer if possible
try:
    df[target_col] = df[target_col].astype(int)
except Exception:
    df[target_col] = df[target_col].astype(str)

print("Cluster counts:")
print(df[target_col].value_counts().sort_index())


# ============================================================
# Prepare X and y
# ============================================================
X = df[numeric_features + categorical_features].copy()
y = df[target_col].copy()

# Replace inf with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Make categorical feature string-like
for col in categorical_features:
    X[col] = X[col].astype("object")

print("X shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# Preprocessing
# ============================================================
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Compatible with different sklearn versions
try:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]
    )
except TypeError:
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ]
    )

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


# ============================================================
# Logistic regression model
# ============================================================
logit_model = LogisticRegression(
    solver="lbfgs",
    max_iter=5000,
    class_weight="balanced",
    random_state=random_state
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("classifier", logit_model)
    ]
)


# ============================================================
# Fit model on full clustered dataset
# ============================================================
model.fit(X, y)

y_pred_full = model.predict(X)
y_prob_full = model.predict_proba(X)

print("\nFull-data classification report:")
print(classification_report(y, y_pred_full))

print("Full-data accuracy:", accuracy_score(y, y_pred_full))


# ============================================================
# Confusion matrix on full data
# This is not independent validation; it shows how well logistic regression
# reproduces the cluster assignment on the fitted dataset.
# ============================================================
classes = model.named_steps["classifier"].classes_

cm = confusion_matrix(y, y_pred_full, labels=classes)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Logistic Regression Confusion Matrix on Full Data")
plt.tight_layout()

cm_path = os.path.join(output_dir, "logistic_regression_full_data_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved full-data confusion matrix to:", cm_path)


# ============================================================
# Get processed feature names
# ============================================================
preprocess_fitted = model.named_steps["preprocess"]

processed_feature_names = []

# Numeric features remain the same after scaling
processed_feature_names.extend(numeric_features)

# One-hot feature names
onehot = (
    preprocess_fitted
    .named_transformers_["cat"]
    .named_steps["onehot"]
)

onehot_feature_names = onehot.get_feature_names_out(categorical_features)
processed_feature_names.extend(onehot_feature_names)

print("\nProcessed feature names:")
print(processed_feature_names)


# ============================================================
# Extract logistic regression coefficients
# ============================================================
classifier = model.named_steps["classifier"]

coef_matrix = classifier.coef_

coef_df = pd.DataFrame(
    coef_matrix,
    columns=processed_feature_names,
    index=[f"cluster_{c}" for c in classifier.classes_]
)

coef_df.index.name = "target_class"

display(coef_df.round(4))

coef_output_path = os.path.join(output_dir, "logistic_regression_coefficients_wide.csv")
coef_df.to_csv(coef_output_path)

print("Saved coefficient table to:", coef_output_path)


# ============================================================
# Long-format coefficient table
# ============================================================
coef_long = (
    coef_df
    .reset_index()
    .melt(
        id_vars="target_class",
        var_name="feature",
        value_name="coefficient"
    )
)

coef_long["abs_coefficient"] = coef_long["coefficient"].abs()

coef_long = coef_long.sort_values(
    ["target_class", "abs_coefficient"],
    ascending=[True, False]
)

display(coef_long.head(30))

coef_long_output_path = os.path.join(output_dir, "logistic_regression_coefficients_long.csv")
coef_long.to_csv(coef_long_output_path, index=False)

print("Saved long coefficient table to:", coef_long_output_path)


# ============================================================
# Top contributing features for each cluster
# ============================================================
top_n = 10

for target_class in coef_long["target_class"].unique():

    temp = (
        coef_long[coef_long["target_class"] == target_class]
        .sort_values("abs_coefficient", ascending=False)
        .head(top_n)
        .copy()
    )

    display(temp)

    plt.figure(figsize=(9, 5))
    plt.barh(temp["feature"], temp["coefficient"])
    plt.axvline(0, linewidth=1)
    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.title(f"Top Logistic Regression Coefficients for {target_class}")
    plt.gca().invert_yaxis()
    plt.tight_layout()

    fig_path = os.path.join(
        output_dir,
        f"logistic_regression_top_coefficients_{target_class}.png"
    )

    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved figure to:", fig_path)


# ============================================================
# Optional: stratified cross-validation
# This checks whether logistic regression can reproduce cluster assignments
# on unseen folds. This is optional because the main purpose is post-hoc
# interpretation rather than independent prediction.
# ============================================================
class_counts = y.value_counts()
min_class_count = class_counts.min()

if min_class_count >= 2:

    n_splits = min(5, min_class_count)

    print(f"\nRunning {n_splits}-fold stratified cross-validation...")

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    y_pred_cv = cross_val_predict(
        model,
        X,
        y,
        cv=cv
    )

    print("\nCross-validation classification report:")
    print(classification_report(y, y_pred_cv))

    print("Cross-validation accuracy:", accuracy_score(y, y_pred_cv))

    cm_cv = confusion_matrix(y, y_pred_cv, labels=classes)

    disp_cv = ConfusionMatrixDisplay(
        confusion_matrix=cm_cv,
        display_labels=classes
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp_cv.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title(f"Logistic Regression Confusion Matrix ({n_splits}-Fold CV)")
    plt.tight_layout()

    cv_cm_path = os.path.join(
        output_dir,
        "logistic_regression_cross_validation_confusion_matrix.png"
    )

    plt.savefig(cv_cm_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved cross-validation confusion matrix to:", cv_cm_path)

else:
    print(
        "\nCross-validation skipped because at least one cluster has fewer than 2 samples."
    )


# ============================================================
# Add predicted class and probabilities to dataframe
# ============================================================
df["logit_pred_cluster"] = y_pred_full

for i, cls in enumerate(classifier.classes_):
    df[f"logit_prob_cluster_{cls}"] = y_prob_full[:, i]

labeled_output_path = os.path.join(
    output_dir,
    "clustered_data_with_logistic_regression_predictions.csv"
)

df.to_csv(labeled_output_path, index=False)

print("Saved data with logistic regression predictions to:", labeled_output_path)


# ============================================================
# Save fitted model
# ============================================================
model_output_path = os.path.join(
    output_dir,
    "logistic_regression_pipeline.joblib"
)

joblib.dump(model, model_output_path)

print("Saved fitted logistic regression pipeline to:", model_output_path)